# Open Mirroring in Microsoft Fabric

This notebook creates a complete Open Mirroring lab. It generates history and incremental Parquet files for two tables, saves them in the attached Lakehouse **Files** section, and then copies them into an Open Mirrored Database landing zone.

You will create and replicate two tables:

- `Customers`
- `Orders`

The lab uses two phases:

1. **History load** - initial table data.
2. **Incremental load** - insert, update, and delete changes for both tables.

---
## Upload This Notebook to Your Fabric Workspace

Before you begin the lab, upload this notebook into your Fabric workspace:

1. Open the [Microsoft Fabric portal](https://app.fabric.microsoft.com).
2. Navigate to your workshop workspace (`<your-workspace>`).
3. Select **New item** → **Import notebook**.
4. Browse to this file (`openmirroring.ipynb`) and upload it.
5. Once imported, open the notebook in your workspace.
6. Attach a **Lakehouse** to the notebook (required for generating and storing Parquet files).
7. You are now ready to follow the lab steps and run the code cells directly in Fabric.

## Part 1 - Create the Open Mirrored Database

Do this in the Fabric portal before running the copy cells:

1. Open your Fabric workspace.
2. Select **New item**.
3. Search for **Mirrored Database**.
4. Create a new mirrored database named `OpenMirroringLab`.
5. Open the mirrored database item.
6. Copy the **Landing zone URL** from the Home page.
7. Paste that URL into the `LANDING_ZONE_URL` variable in the next cell.

The landing zone is where this notebook copies the table folders, metadata, history files, and incremental files.

## Part 2 - Attach a Lakehouse

Attach a Lakehouse to this notebook. The notebook writes generated files into the Lakehouse **Files** section first:

```text
Files/open-mirroring-lab
```

After you inspect the files, later cells copy them into the Open Mirroring landing zone.

## Part 3 - Configuration

Paste the landing zone URL from the Open Mirrored Database Home page. You can paste either the `https://onelake.dfs.fabric.microsoft.com/...` URL shown in Fabric or an `abfss://...` URL.

In [ ]:
OPEN_MIRROR_ITEM_NAME = "OpenMirroringLab"

# Paste the landing zone URL from the Open Mirrored Database Home page.
# Example https form:
# LANDING_ZONE_URL = "https://onelake.dfs.fabric.microsoft.com/<workspace>/<mirrored database>/Files/LandingZone"
# Example abfss form:
# LANDING_ZONE_URL = "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<mirrored database>.MirroredDatabase/Files/LandingZone"
LANDING_ZONE_URL = ""

LAB_ROOT = "open-mirroring-lab"


def normalize_landing_zone_url(url: str) -> str:
    url = url.strip().rstrip("/")
    if not url:
        raise ValueError("Paste the Open Mirroring landing zone URL into LANDING_ZONE_URL.")
    if url.startswith("abfss://"):
        return url
    prefix = "https://onelake.dfs.fabric.microsoft.com/"
    if url.startswith(prefix):
        rest = url[len(prefix):]
        workspace, item_path = rest.split("/", 1)
        return f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{item_path}".rstrip("/")
    raise ValueError("Landing zone URL must start with https://onelake.dfs.fabric.microsoft.com/ or abfss://")


LANDING_ZONE_ABFSS = normalize_landing_zone_url(LANDING_ZONE_URL)
print("Landing zone:", LANDING_ZONE_ABFSS)

## Part 4 - Generate history and incremental files in Lakehouse Files

This cell creates all lab files under the attached Lakehouse Files section:

```text
Files/open-mirroring-lab/history/...
Files/open-mirroring-lab/incremental/...
```

The history files do **not** include `__rowMarker__`. The incremental files include `__rowMarker__` as the final column.

In [ ]:
from pathlib import Path
import json
import os
import shutil

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "-q"])
    import pyarrow as pa
    import pyarrow.parquet as pq

FILES_ROOT = Path("/lakehouse/default/Files") / LAB_ROOT
if not Path("/lakehouse/default/Files").exists():
    raise RuntimeError("Attach a Lakehouse to this notebook before running this cell.")

TABLES = {
    "Customers": {
        "keyColumns": ["CustomerID"],
        "schema": [
            ("CustomerID", pa.string()),
            ("CustomerName", pa.string()),
            ("City", pa.string()),
            ("Status", pa.string()),
            ("LastUpdated", pa.string()),
        ],
        "history": [
            {"CustomerID":"C001", "CustomerName":"Contoso Retail", "City":"Seattle", "Status":"Active", "LastUpdated":"2026-08-01T09:00:00Z"},
            {"CustomerID":"C002", "CustomerName":"Fabrikam Labs", "City":"Portland", "Status":"Active", "LastUpdated":"2026-08-01T09:05:00Z"},
            {"CustomerID":"C003", "CustomerName":"Adventure Works", "City":"Vancouver", "Status":"Active", "LastUpdated":"2026-08-01T09:10:00Z"},
        ],
        "incremental": [
            {"CustomerID":"C002", "CustomerName":"Fabrikam Labs", "City":"Bellevue", "Status":"Active", "LastUpdated":"2026-08-02T09:00:00Z", "__rowMarker__":1},
            {"CustomerID":"C004", "CustomerName":"Northwind Traders", "City":"Redmond", "Status":"Active", "LastUpdated":"2026-08-02T09:05:00Z", "__rowMarker__":0},
            {"CustomerID":"C003", "CustomerName":"Adventure Works", "City":"Vancouver", "Status":"Active", "LastUpdated":"2026-08-02T09:10:00Z", "__rowMarker__":2},
        ],
    },
    "Orders": {
        "keyColumns": ["OrderID"],
        "schema": [
            ("OrderID", pa.string()),
            ("CustomerID", pa.string()),
            ("Product", pa.string()),
            ("Quantity", pa.int64()),
            ("Amount", pa.float64()),
            ("OrderStatus", pa.string()),
            ("LastUpdated", pa.string()),
        ],
        "history": [
            {"OrderID":"O1001", "CustomerID":"C001", "Product":"Widget", "Quantity":2, "Amount":199.90, "OrderStatus":"Open", "LastUpdated":"2026-08-01T10:00:00Z"},
            {"OrderID":"O1002", "CustomerID":"C002", "Product":"Laptop", "Quantity":1, "Amount":1299.00, "OrderStatus":"Open", "LastUpdated":"2026-08-01T10:05:00Z"},
            {"OrderID":"O1003", "CustomerID":"C003", "Product":"Support Plan", "Quantity":1, "Amount":499.00, "OrderStatus":"Open", "LastUpdated":"2026-08-01T10:10:00Z"},
        ],
        "incremental": [
            {"OrderID":"O1002", "CustomerID":"C002", "Product":"Laptop", "Quantity":1, "Amount":1299.00, "OrderStatus":"Shipped", "LastUpdated":"2026-08-02T10:00:00Z", "__rowMarker__":1},
            {"OrderID":"O1004", "CustomerID":"C004", "Product":"Cloud Backup", "Quantity":3, "Amount":299.97, "OrderStatus":"Open", "LastUpdated":"2026-08-02T10:05:00Z", "__rowMarker__":0},
            {"OrderID":"O1003", "CustomerID":"C003", "Product":"Support Plan", "Quantity":1, "Amount":499.00, "OrderStatus":"Open", "LastUpdated":"2026-08-02T10:10:00Z", "__rowMarker__":2},
        ],
    },
}


def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def write_parquet(path: Path, rows: list[dict], schema_spec: list[tuple[str, pa.DataType]], include_row_marker: bool):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = [pa.field(name, dtype) for name, dtype in schema_spec]
    if include_row_marker:
        fields.append(pa.field("__rowMarker__", pa.int32()))
    schema = pa.schema(fields)
    columns = {field.name: [row.get(field.name) for row in rows] for field in schema}
    table = pa.table(columns, schema=schema)
    pq.write_table(table, path)

# Recreate the lab files area for a clean run.
if FILES_ROOT.exists():
    shutil.rmtree(FILES_ROOT)

partner_events = {
    "partnerName": "Fabric DI Workshop",
    "sourceInfo": {
        "sourceType": "NotebookGeneratedFiles",
        "sourceVersion": "1.0",
        "additionalInformation": {"lab": "Open Mirroring"}
    }
}
write_json(FILES_ROOT / "history" / "_partnerEvents.json", partner_events)

for table_name, spec in TABLES.items():
    write_json(FILES_ROOT / "history" / table_name / "_metadata.json", {"keyColumns": spec["keyColumns"]})
    write_parquet(
        FILES_ROOT / "history" / table_name / "00000000000000000001.parquet",
        spec["history"],
        spec["schema"],
        include_row_marker=False,
    )
    write_parquet(
        FILES_ROOT / "incremental" / table_name / "00000000000000000002.parquet",
        spec["incremental"],
        spec["schema"],
        include_row_marker=True,
    )

print(f"Generated files under Files/{LAB_ROOT}")
for path in sorted(FILES_ROOT.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(Path("/lakehouse/default/Files")))

## Part 5 - Inspect staged files

This cell lists the files saved in the Lakehouse Files section. Open the Lakehouse in the Fabric portal and browse to `Files/open-mirroring-lab` to see the same structure.

In [ ]:
for phase in ["history", "incremental"]:
    print(f"\n{phase.upper()} FILES")
    for path in sorted((FILES_ROOT / phase).rglob("*")):
        if path.is_file():
            print(" -", path.relative_to(FILES_ROOT))

## Part 6 - Copy history files to the Open Mirroring landing zone

This copies `_metadata.json`, `_partnerEvents.json`, and the `00000000000000000001.parquet` history files from Lakehouse Files into the Open Mirroring landing zone.

After running this cell, go to the Open Mirrored Database item and start mirroring with **Mirror all data** selected.

In [ ]:
def get_fabric_fs():
    try:
        return notebookutils.fs
    except NameError:
        return mssparkutils.fs

fs = get_fabric_fs()


def cp_local_to_landing(local_path: Path, landing_relative_path: str):
    target = f"{LANDING_ZONE_ABFSS}/{landing_relative_path}".replace("//", "/").replace("abfss:/", "abfss://")
    parent = target.rsplit("/", 1)[0]
    fs.mkdirs(parent)
    fs.cp(f"file:{local_path}", target)
    print("Copied", local_path.relative_to(FILES_ROOT), "->", target)

# Optional partner events at mirrored database landing zone root.
cp_local_to_landing(FILES_ROOT / "history" / "_partnerEvents.json", "_partnerEvents.json")

for table_name in TABLES:
    cp_local_to_landing(FILES_ROOT / "history" / table_name / "_metadata.json", f"{table_name}/_metadata.json")
    cp_local_to_landing(
        FILES_ROOT / "history" / table_name / "00000000000000000001.parquet",
        f"{table_name}/00000000000000000001.parquet",
    )

print("History files copied. Start mirroring and monitor Customers and Orders.")

## Part 7 - Monitor history load

In the Open Mirrored Database item:

1. Select **Mirror database** if mirroring has not started.
2. Leave **Mirror all data** selected.
3. Select **Monitor replication**.
4. Wait until `Customers` and `Orders` show as running and refreshed.

Then run the optional validation cell below.

In [ ]:
# Optional validation after history load has completed.
# If this fails immediately after copying files, wait a minute and rerun.
for table_name in ["Customers", "Orders"]:
    print(f"\n{table_name}")
    display(spark.read.synapsesql(f"{OPEN_MIRROR_ITEM_NAME}.{table_name}").orderBy(table_name[:-1] + "ID" if table_name.endswith('s') else table_name + "ID"))

## Part 8 - Copy incremental files to the Open Mirroring landing zone

This copies `00000000000000000002.parquet` for both tables. Each file contains an insert, update, and delete represented with `__rowMarker__`.

Run this only after the history load is visible in the mirrored database.

In [ ]:
for table_name in TABLES:
    cp_local_to_landing(
        FILES_ROOT / "incremental" / table_name / "00000000000000000002.parquet",
        f"{table_name}/00000000000000000002.parquet",
    )

print("Incremental files copied. Monitor replication until Customers and Orders refresh again.")

## Part 9 - Validate incremental changes

After the monitor page shows a new refresh timestamp, run this validation cell. Expected final state:

- `Customers`: C001, C002 updated to Bellevue, C004 inserted, C003 deleted.
- `Orders`: O1001, O1002 updated to Shipped, O1004 inserted, O1003 deleted.

In [ ]:
print("Customers final state")
display(spark.read.synapsesql(f"{OPEN_MIRROR_ITEM_NAME}.Customers").orderBy("CustomerID"))

print("Orders final state")
display(spark.read.synapsesql(f"{OPEN_MIRROR_ITEM_NAME}.Orders").orderBy("OrderID"))

## Troubleshooting

| Symptom | What to check |
| --- | --- |
| Tables do not appear | Confirm history files and `_metadata.json` were copied under `Files/LandingZone/<table>`. |
| Updates/deletes do not apply | Confirm `_metadata.json` has `keyColumns`, and incremental Parquet has `__rowMarker__` as the final column. |
| Notebook copy fails | Confirm the landing zone URL is pasted correctly and you have access to the Open Mirrored Database item. |
| Validation query fails | Wait for replication to finish, then rerun the validation cell. |

## Reference

- [Tutorial: Configure Microsoft Fabric open mirrored databases](https://learn.microsoft.com/en-us/fabric/mirroring/open-mirroring-tutorial)
- [Open Mirroring landing zone requirements and format](https://learn.microsoft.com/en-us/fabric/mirroring/open-mirroring-landing-zone-format)